In [ ]:
# Configuration - works both locally and on Databricks
import os

# Detect environment
IS_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

if IS_DATABRICKS:
    # Databricks widgets for parameterization
    dbutils.widgets.text('cur_bucket', 'flex-cur-data')
    dbutils.widgets.text('year', '2026')
    dbutils.widgets.text('month', '05')
    CUR_BUCKET = dbutils.widgets.get('cur_bucket')
    YEAR = dbutils.widgets.get('year')
    MONTH = dbutils.widgets.get('month')
    S3_PATH = f's3://{CUR_BUCKET}/raw/{YEAR}/{MONTH}/'
    STAGING_PATH = f's3://{CUR_BUCKET}/staging/{YEAR}/{MONTH}/'
else:
    # Local with LocalStack
    CUR_BUCKET = os.environ.get('CUR_BUCKET', 'flex-cur-data')
    S3_ENDPOINT = os.environ.get('AWS_ENDPOINT_URL', 'http://localhost:4566')
    YEAR = '2026'
    MONTH = '05'
    S3_PATH = f's3a://{CUR_BUCKET}/raw/{YEAR}/{MONTH}/'
    STAGING_PATH = f's3a://{CUR_BUCKET}/staging/{YEAR}/{MONTH}/'

print(f'Environment: {"Databricks" if IS_DATABRICKS else "Local"}')
print(f'Reading from: {S3_PATH}')
print(f'Writing to: {STAGING_PATH}')

In [ ]:
# Initialize Spark Session
from pyspark.sql import SparkSession

if not IS_DATABRICKS:
    spark = (SparkSession.builder
        .master('local[*]')
        .appName('Flex-CUR-Ingest')
        .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262')
        .config('spark.hadoop.fs.s3a.endpoint', S3_ENDPOINT)
        .config('spark.hadoop.fs.s3a.access.key', 'test')
        .config('spark.hadoop.fs.s3a.secret.key', 'test')
        .config('spark.hadoop.fs.s3a.path.style.access', 'true')
        .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
        .config('spark.driver.memory', '4g')
        .getOrCreate())
else:
    spark = SparkSession.builder.getOrCreate()

print(f'Spark version: {spark.version}')

In [ ]:
# Read raw CUR Parquet files from S3
from pyspark.sql import functions as F
from pyspark.sql.types import *

# For local dev without S3, fallback to local parquet files
try:
    raw_df = spark.read.parquet(S3_PATH)
    print(f'Read from S3: {S3_PATH}')
except Exception as e:
    print(f'S3 read failed ({e}), falling back to local files...')
    LOCAL_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'cur_raw', f'cur_{YEAR}_{MONTH}.parquet')
    raw_df = spark.read.parquet(LOCAL_PATH)
    print(f'Read from local: {LOCAL_PATH}')

print(f'Raw records: {raw_df.count():,}')
print(f'Columns: {len(raw_df.columns)}')
raw_df.printSchema()

In [ ]:
# Schema Validation - ensure required CUR columns exist
REQUIRED_COLUMNS = [
    'identity/LineItemId',
    'lineItem/UsageStartDate',
    'lineItem/ProductCode',
    'lineItem/UsageAmount',
    'lineItem/UnblendedCost',
    'lineItem/BlendedCost',
    'custom/BusinessUnitId',
    'product/region',
]

missing = [col for col in REQUIRED_COLUMNS if col not in raw_df.columns]
if missing:
    raise ValueError(f'Missing required CUR columns: {missing}')

print('✅ Schema validation passed - all required columns present')

In [ ]:
# Data Cleaning & Deduplication

# 1. Drop exact duplicates by LineItemId
deduped_df = raw_df.dropDuplicates(['identity/LineItemId'])
dupes_removed = raw_df.count() - deduped_df.count()
print(f'Duplicates removed: {dupes_removed:,}')

# 2. Filter out zero-cost items (noise)
clean_df = deduped_df.filter(
    (F.col('lineItem/UnblendedCost') > 0) | 
    (F.col('lineItem/UsageAmount') > 0)
)
zero_removed = deduped_df.count() - clean_df.count()
print(f'Zero-cost records filtered: {zero_removed:,}')

# 3. Cast types
clean_df = (clean_df
    .withColumn('usage_date', F.to_timestamp('lineItem/UsageStartDate'))
    .withColumn('cost', F.col('lineItem/UnblendedCost').cast('double'))
    .withColumn('blended_cost', F.col('lineItem/BlendedCost').cast('double'))
    .withColumn('usage_amount', F.col('lineItem/UsageAmount').cast('double'))
)

# 4. Add ingestion metadata
clean_df = (clean_df
    .withColumn('_ingested_at', F.current_timestamp())
    .withColumn('_source_file', F.input_file_name())
)

print(f'\n✅ Clean records ready: {clean_df.count():,}')

In [ ]:
# Data Quality Checks

# Check for null BU IDs (would break multi-tenant routing)
null_bu = clean_df.filter(F.col('custom/BusinessUnitId').isNull()).count()
assert null_bu == 0, f'Found {null_bu} records with null BusinessUnitId!'

# Check date range sanity
date_stats = clean_df.agg(
    F.min('usage_date').alias('min_date'),
    F.max('usage_date').alias('max_date'),
    F.countDistinct('custom/BusinessUnitId').alias('bu_count'),
    F.sum('cost').alias('total_cost'),
).collect()[0]

print(f'Date range: {date_stats.min_date} to {date_stats.max_date}')
print(f'Business Units: {date_stats.bu_count}')
print(f'Total cost: ${date_stats.total_cost:,.2f}')
print('\n✅ Data quality checks passed')

In [ ]:
# Write to Staging (partitioned by BU and date)

try:
    (clean_df
        .repartition('custom/BusinessUnitId')
        .write
        .mode('overwrite')
        .partitionBy('custom/BusinessUnitId', 'lineItem/ProductCode')
        .parquet(STAGING_PATH)
    )
    print(f'✅ Staging written to: {STAGING_PATH}')
except Exception as e:
    # Fallback to local
    local_staging = os.path.join(os.path.dirname(os.getcwd()), 'data', 'staging', f'{YEAR}_{MONTH}')
    os.makedirs(local_staging, exist_ok=True)
    (clean_df
        .repartition('custom/BusinessUnitId')
        .write
        .mode('overwrite')
        .partitionBy('custom/BusinessUnitId', 'lineItem/ProductCode')
        .parquet(local_staging)
    )
    print(f'✅ Staging written locally to: {local_staging}')

print(f'\nIngestion complete for {YEAR}-{MONTH}')